# Notebook 2: Link Discharge Notes and Chest X-Ray Reports to Long-Stay Cohort

## Objective
Link discharge summaries and chest X-ray (CXR) reports to the long-stay cohort (LOS ≥15 days) provided by partner for multimodal readmission prediction.

## Input Data
- `long_los_cohort.csv` - Partner's labeled cohort (17,940 admissions, LOS ≥15 days)
- `discharge.csv` - Discharge summaries from MIMIC-IV-Note (331,793 notes)
- `radiology.csv` - Radiology reports from MIMIC-IV-Note (2.3M reports)

## Process
1. **Load partner's cohort** - 17,940 long-stay admissions with clinical condition flags
2. **Link discharge summaries** - Match discharge notes (type='DS') to admissions
3. **Filter radiology to chest X-rays** - Identify CXR reports (exclude CT scans)
4. **Select final CXR** - Keep most recent CXR before discharge per admission
5. **Merge notes with cohort** - Create combined dataset with both note types
6. **Calculate text metrics** - Assess character counts and token estimates

## Output
**File:** `long_los_cohort_with_notes.csv`  
**Size:** 315.2 MB  
**Rows:** 17,940 admissions  
**Columns:** 30 features

### Coverage Statistics:
- Admissions with discharge notes: **17,940 (100.0%)**
- Admissions with chest X-rays: **16,930 (94.4%)**
- Admissions with BOTH notes: **16,930 (94.4%)**
- Discharge notes only: 1,010 (5.6%)

### Cohort Characteristics:
- **Readmission rate:** 24.49% overall, 24.78% in CXR subset (higher than general population)
- **Mean age:** 62.3 years (range: 18-100)
- **Mean LOS:** 24.8 days (range: 14-296)
- **Clinical conditions:** 56.9% cardiorenal, 38.4% AKI, 27.5% heart failure, 17.7% sepsis

### Text Statistics:
- **Average discharge note:** 17,172 chars (~4,293 tokens)
- **Average CXR report:** 1,028 chars (~257 tokens)
- **Average combined:** 18,347 chars (~4,587 tokens)
- **Notes exceeding 8K token limit:** 520 (3.1%) - minimal truncation needed

### Quality Checks:
- 100% discharge note coverage (all admissions have discharge summaries)
- 94.4% CXR coverage (16,930/17,940 admissions)
- Only 20 post-discharge CXRs excluded (99.99% temporal validity)
- Average 8.3 CXRs per admission (kept most recent before discharge)

## Key Findings
Patients with chest X-rays show higher readmission risk (24.78% vs 19.70%), suggesting CXR availability correlates with illness severity. BioClinical-ModernBERT's 8,192 token context accommodates 96.9% of combined notes without truncation. The long-stay cohort represents complex patients with high comorbidity burden (64.8% have cardiorenal/sepsis conditions).

## Next Step
Notebook 4B will encode both discharge summaries and CXR reports using BioClinical-ModernBERT to create dense embeddings for readmission prediction.

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

print("Libraries imported")

Libraries imported


In [3]:
# Update these paths to match your Google Drive structure
BASE_DIR = Path('/content/drive/MyDrive/MIDS/w266/Final Project')
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'output'

# Input files
COHORT_FILE = DATA_DIR / 'long_los_cohort.csv'  # Partner's cohort
DISCHARGE_FILE = DATA_DIR / 'discharge.csv'
RADIOLOGY_FILE = DATA_DIR / 'radiology.csv'

# Output file
OUTPUT_FILE = OUTPUT_DIR / 'long_los_cohort_with_notes.csv'

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"\nInput files:")
print(f"  Cohort: {COHORT_FILE}")
print(f"  Discharge: {DISCHARGE_FILE}")
print(f"  Radiology: {RADIOLOGY_FILE}")
print(f"\nOutput file:")
print(f"  {OUTPUT_FILE}")

Data directory: /content/drive/MyDrive/MIDS/w266/Final Project/data
Output directory: /content/drive/MyDrive/MIDS/w266/Final Project/output

Input files:
  Cohort: /content/drive/MyDrive/MIDS/w266/Final Project/data/long_los_cohort.csv
  Discharge: /content/drive/MyDrive/MIDS/w266/Final Project/data/discharge.csv
  Radiology: /content/drive/MyDrive/MIDS/w266/Final Project/data/radiology.csv

Output file:
  /content/drive/MyDrive/MIDS/w266/Final Project/output/long_los_cohort_with_notes.csv


In [4]:
print("Loading partner's cohort...")
cohort = pd.read_csv(COHORT_FILE)

print(f"\n{'='*60}")
print("COHORT OVERVIEW")
print(f"{'='*60}")
print(f"\nTotal admissions: {len(cohort):,}")
print(f"Total columns: {len(cohort.columns)}")
print(f"\nColumn names:")
for i, col in enumerate(cohort.columns, 1):
    print(f"  {i:2d}. {col}")

print(f"\n{'='*60}")
print("FIRST 3 ROWS")
print(f"{'='*60}")
print(cohort.head(3))

Loading partner's cohort...

COHORT OVERVIEW

Total admissions: 17,940
Total columns: 20

Column names:
   1. subject_id
   2. hadm_id
   3. admittime
   4. dischtime
   5. length_of_stay_days
   6. readmitted_within_window
   7. readmission_gap_in_days
   8. admission_type
   9. discharge_location
  10. age_at_admit
  11. gender
  12. race
  13. is_cardiorenal_long
  14. has_acute_kidney_injury
  15. has_heart_failure
  16. has_hyponatremia
  17. has_posthemorrhagic_anemia
  18. has_sepsis
  19. has_any_cardiorenal_sepsis
  20. has_aki_and_hf

FIRST 3 ROWS
   subject_id   hadm_id            admittime            dischtime  \
0    10001338  22119639  2138-05-09 19:47:00  2138-05-27 15:40:00   
1    10002155  23822395  2129-08-04 12:44:00  2129-08-18 16:53:00   
2    10002428  28662225  2156-04-12 14:16:00  2156-04-29 16:26:00   

   length_of_stay_days  readmitted_within_window  readmission_gap_in_days  \
0            17.828472                      True                 6.761806   
1    

In [5]:
print(f"{'='*60}")
print("DATA TYPES AND MISSING VALUES")
print(f"{'='*60}")

# Check data types
print("\nColumn data types:")
print(cohort.dtypes)

print(f"\n{'='*60}")
print("MISSING VALUES")
print(f"{'='*60}")
missing = cohort.isnull().sum()
missing_pct = (cohort.isnull().sum() / len(cohort) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing_Count': missing,
    'Missing_Percent': missing_pct
})
print(missing_df[missing_df['Missing_Count'] > 0])

if missing_df['Missing_Count'].sum() == 0:
    print("✓ No missing values!")

DATA TYPES AND MISSING VALUES

Column data types:
subject_id                      int64
hadm_id                         int64
admittime                      object
dischtime                      object
length_of_stay_days           float64
readmitted_within_window         bool
readmission_gap_in_days       float64
admission_type                 object
discharge_location             object
age_at_admit                    int64
gender                         object
race                           object
is_cardiorenal_long              bool
has_acute_kidney_injury          bool
has_heart_failure                bool
has_hyponatremia                 bool
has_posthemorrhagic_anemia       bool
has_sepsis                       bool
has_any_cardiorenal_sepsis       bool
has_aki_and_hf                   bool
dtype: object

MISSING VALUES
                         Missing_Count  Missing_Percent
readmission_gap_in_days           7446            41.51


In [6]:
print(f"{'='*60}")
print("COHORT STATISTICS")
print(f"{'='*60}")

print(f"\nCohort size: {len(cohort):,} admissions")
print(f"Unique patients: {cohort['subject_id'].nunique():,}")
print(f"Unique admissions: {cohort['hadm_id'].nunique():,}")

print(f"\n{'='*60}")
print("READMISSION LABEL")
print(f"{'='*60}")
print(cohort['readmitted_within_window'].value_counts())
print(f"\nReadmission rate: {cohort['readmitted_within_window'].mean()*100:.2f}%")

print(f"\n{'='*60}")
print("DEMOGRAPHICS")
print(f"{'='*60}")
print(f"\nAge:")
print(f"  Mean: {cohort['age_at_admit'].mean():.1f} years")
print(f"  Median: {cohort['age_at_admit'].median():.1f} years")
print(f"  Range: {cohort['age_at_admit'].min():.0f} - {cohort['age_at_admit'].max():.0f} years")

print(f"\nGender:")
print(cohort['gender'].value_counts())

print(f"\nRace (top 5):")
print(cohort['race'].value_counts().head())

print(f"\n{'='*60}")
print("LENGTH OF STAY")
print(f"{'='*60}")
print(f"Mean: {cohort['length_of_stay_days'].mean():.1f} days")
print(f"Median: {cohort['length_of_stay_days'].median():.1f} days")
print(f"Min: {cohort['length_of_stay_days'].min():.1f} days")
print(f"Max: {cohort['length_of_stay_days'].max():.1f} days")
print(f"\nLOS ≥15 days: {(cohort['length_of_stay_days'] >= 15).sum():,} ({(cohort['length_of_stay_days'] >= 15).sum()/len(cohort)*100:.1f}%)")

print(f"\n{'='*60}")
print("CLINICAL CONDITIONS")
print(f"{'='*60}")
condition_cols = [col for col in cohort.columns if col.startswith('has_') or col.startswith('is_')]
for col in condition_cols:
    count = cohort[col].sum()
    pct = count / len(cohort) * 100
    print(f"{col}: {count:,} ({pct:.1f}%)")

COHORT STATISTICS

Cohort size: 17,940 admissions
Unique patients: 14,262
Unique admissions: 17,940

READMISSION LABEL
readmitted_within_window
False    13546
True      4394
Name: count, dtype: int64

Readmission rate: 24.49%

DEMOGRAPHICS

Age:
  Mean: 62.3 years
  Median: 64.0 years
  Range: 18 - 100 years

Gender:
gender
M    9890
F    8050
Name: count, dtype: int64

Race (top 5):
race
WHITE                     11621
BLACK/AFRICAN AMERICAN     1946
UNKNOWN                    1134
OTHER                       581
WHITE - OTHER EUROPEAN      334
Name: count, dtype: int64

LENGTH OF STAY
Mean: 24.8 days
Median: 20.0 days
Min: 14.0 days
Max: 296.0 days

LOS ≥15 days: 15,659 (87.3%)

CLINICAL CONDITIONS
is_cardiorenal_long: 10,204 (56.9%)
has_acute_kidney_injury: 6,890 (38.4%)
has_heart_failure: 4,928 (27.5%)
has_hyponatremia: 2,963 (16.5%)
has_posthemorrhagic_anemia: 3,073 (17.1%)
has_sepsis: 3,183 (17.7%)
has_any_cardiorenal_sepsis: 11,626 (64.8%)
has_aki_and_hf: 2,922 (16.3%)


In [7]:
print("Loading discharge summaries...")
print("This may take a few minutes...")

discharge = pd.read_csv(DISCHARGE_FILE)

print(f"\nDischarge notes loaded!")
print(f"Total notes: {len(discharge):,}")
print(f"Columns: {discharge.columns.tolist()}")
print(f"\nFirst row:")
print(discharge.head(1))

Loading discharge summaries...
This may take a few minutes...

Discharge notes loaded!
Total notes: 331,793
Columns: ['note_id', 'subject_id', 'hadm_id', 'note_type', 'note_seq', 'charttime', 'storetime', 'text']

First row:
          note_id  subject_id   hadm_id note_type  note_seq  \
0  10000032-DS-21    10000032  22595853        DS        21   

             charttime            storetime  \
0  2180-05-07 00:00:00  2180-05-09 15:26:00   

                                                text  
0   \nName:  ___                     Unit No:   _...  


In [8]:
print(f"\n{'='*60}")
print("LINKING DISCHARGE NOTES TO COHORT")
print(f"{'='*60}")

# Filter discharge notes to only 'DS' type (Discharge Summary)
print(f"\nFiltering to Discharge Summaries only...")
discharge_ds = discharge[discharge['note_type'] == 'DS'].copy()
print(f"Discharge summaries: {len(discharge_ds):,}")
print(f"Unique admissions with DS: {discharge_ds['hadm_id'].nunique():,}")

# Check for duplicates (multiple discharge summaries per admission)
duplicates = discharge_ds.groupby('hadm_id').size()
print(f"\nAdmissions with multiple discharge summaries: {(duplicates > 1).sum():,}")
if (duplicates > 1).sum() > 0:
    print(f"Max discharge summaries per admission: {duplicates.max()}")

# For admissions with multiple, keep the last one (most recent charttime)
print(f"\nKeeping most recent discharge summary per admission...")
discharge_ds = discharge_ds.sort_values('charttime').groupby('hadm_id').tail(1)
print(f"After deduplication: {len(discharge_ds):,} discharge summaries")

# Rename columns to avoid conflicts
discharge_ds = discharge_ds.rename(columns={
    'text': 'discharge_text',
    'charttime': 'discharge_charttime',
    'note_id': 'discharge_note_id'
})

# Merge with cohort
print(f"\nMerging discharge notes with cohort...")
cohort_with_discharge = cohort.merge(
    discharge_ds[['hadm_id', 'discharge_text', 'discharge_charttime', 'discharge_note_id']],
    on='hadm_id',
    how='left'
)

print(f"\nResults:")
print(f"  Cohort admissions: {len(cohort):,}")
print(f"  Admissions with discharge notes: {cohort_with_discharge['discharge_text'].notna().sum():,}")
print(f"  Admissions WITHOUT discharge notes: {cohort_with_discharge['discharge_text'].isna().sum():,}")
print(f"  Coverage: {cohort_with_discharge['discharge_text'].notna().sum()/len(cohort)*100:.1f}%")

print(f"\nReadmission rate in cohort with discharge notes: {cohort_with_discharge[cohort_with_discharge['discharge_text'].notna()]['readmitted_within_window'].mean()*100:.2f}%")


LINKING DISCHARGE NOTES TO COHORT

Filtering to Discharge Summaries only...
Discharge summaries: 331,793
Unique admissions with DS: 331,793

Admissions with multiple discharge summaries: 0

Keeping most recent discharge summary per admission...
After deduplication: 331,793 discharge summaries

Merging discharge notes with cohort...

Results:
  Cohort admissions: 17,940
  Admissions with discharge notes: 17,940
  Admissions WITHOUT discharge notes: 0
  Coverage: 100.0%

Readmission rate in cohort with discharge notes: 24.49%


In [9]:
print(f"\n{'='*60}")
print("LOADING RADIOLOGY REPORTS")
print(f"{'='*60}")

print("Loading radiology.csv...")
print("This may take several minutes (large file ~2.3M reports)...")

radiology = pd.read_csv(RADIOLOGY_FILE)

print(f"\nRadiology reports loaded!")
print(f"Total reports: {len(radiology):,}")
print(f"Columns: {radiology.columns.tolist()}")
print(f"\nFirst row:")
print(radiology.head(1))


LOADING RADIOLOGY REPORTS
Loading radiology.csv...
This may take several minutes (large file ~2.3M reports)...

Radiology reports loaded!
Total reports: 2,321,355
Columns: ['note_id', 'subject_id', 'hadm_id', 'note_type', 'note_seq', 'charttime', 'storetime', 'text']

First row:
          note_id  subject_id     hadm_id note_type  note_seq  \
0  10000032-RR-14    10000032  22595853.0        RR        14   

             charttime            storetime  \
0  2180-05-06 21:19:00  2180-05-06 23:32:00   

                                                text  
0  EXAMINATION:  CHEST (PA AND LAT)\n\nINDICATION...  


In [10]:
print(f"\n{'='*60}")
print("FILTERING TO CHEST X-RAYS")
print(f"{'='*60}")

# Filter radiology reports to those in our cohort
print(f"Filtering radiology to cohort admissions...")
radiology_cohort = radiology[radiology['hadm_id'].isin(cohort['hadm_id'])].copy()
print(f"Radiology reports for cohort admissions: {len(radiology_cohort):,}")
print(f"Unique admissions with radiology: {radiology_cohort['hadm_id'].nunique():,}")

# Identify chest X-rays
print(f"\nIdentifying chest X-ray reports...")
chest_keywords = ['chest', 'thorax', 'cxr', 'portable chest', 'ap chest']

# Create boolean mask for chest reports
is_chest = radiology_cohort['text'].str.lower().str.contains('|'.join(chest_keywords), na=False)
print(f"Reports containing chest keywords: {is_chest.sum():,} ({is_chest.sum()/len(radiology_cohort)*100:.1f}%)")

# More specific: actual chest X-ray (not CT)
is_cxr = (
    radiology_cohort['text'].str.lower().str.contains('chest', na=False) &
    ~radiology_cohort['text'].str.lower().str.contains('ct chest|computed tomography', na=False)
)
print(f"Chest X-rays (excluding CT): {is_cxr.sum():,} ({is_cxr.sum()/len(radiology_cohort)*100:.1f}%)")

# Filter to chest X-rays
cxr_reports = radiology_cohort[is_cxr].copy()
print(f"\nChest X-ray reports: {len(cxr_reports):,}")
print(f"Unique admissions with CXR: {cxr_reports['hadm_id'].nunique():,}")

# Check for multiple CXRs per admission
cxr_per_admission = cxr_reports.groupby('hadm_id').size()
print(f"\nAdmissions with multiple CXRs: {(cxr_per_admission > 1).sum():,}")
print(f"Average CXRs per admission: {cxr_per_admission.mean():.1f}")
print(f"Max CXRs per admission: {cxr_per_admission.max()}")

# Distribution of CXRs per admission
print(f"\nCXR count distribution:")
print(cxr_per_admission.value_counts().head(10))


FILTERING TO CHEST X-RAYS
Filtering radiology to cohort admissions...
Radiology reports for cohort admissions: 250,694
Unique admissions with radiology: 17,940

Identifying chest X-ray reports...
Reports containing chest keywords: 155,263 (61.9%)
Chest X-rays (excluding CT): 139,855 (55.8%)

Chest X-ray reports: 139,855
Unique admissions with CXR: 16,930

Admissions with multiple CXRs: 14,863
Average CXRs per admission: 8.3
Max CXRs per admission: 109

CXR count distribution:
1     2067
2     1981
3     1815
4     1495
5     1264
6     1045
7      848
8      788
9      701
10     560
Name: count, dtype: int64


In [11]:
print(f"\n{'='*60}")
print("SELECTING FINAL CHEST X-RAY PER ADMISSION")
print(f"{'='*60}")

# Convert charttime to datetime
cxr_reports['charttime'] = pd.to_datetime(cxr_reports['charttime'])

# Merge with cohort to get discharge times
cxr_with_disch = cxr_reports.merge(
    cohort[['hadm_id', 'dischtime']],
    on='hadm_id',
    how='left'
)

# Convert dischtime to datetime
cxr_with_disch['dischtime'] = pd.to_datetime(cxr_with_disch['dischtime'])

# Filter to CXRs before discharge
print(f"Filtering to CXRs before discharge...")
cxr_before_disch = cxr_with_disch[cxr_with_disch['charttime'] <= cxr_with_disch['dischtime']].copy()
print(f"CXRs before discharge: {len(cxr_before_disch):,}")
print(f"CXRs after discharge (excluded): {len(cxr_with_disch) - len(cxr_before_disch):,}")

# For each admission, keep the MOST RECENT CXR before discharge
print(f"\nKeeping most recent CXR per admission...")
cxr_final = cxr_before_disch.sort_values('charttime').groupby('hadm_id').tail(1).copy()
print(f"Final CXR reports: {len(cxr_final):,}")
print(f"Unique admissions with final CXR: {cxr_final['hadm_id'].nunique():,}")

# Rename columns
cxr_final = cxr_final.rename(columns={
    'text': 'radiology_text',
    'charttime': 'radiology_charttime',
    'note_id': 'radiology_note_id'
})

# Drop dischtime (already in cohort)
cxr_final = cxr_final.drop(columns=['dischtime'])

print(f"\n✓ Ready to merge CXR reports with cohort")


SELECTING FINAL CHEST X-RAY PER ADMISSION
Filtering to CXRs before discharge...
CXRs before discharge: 139,835
CXRs after discharge (excluded): 20

Keeping most recent CXR per admission...
Final CXR reports: 16,930
Unique admissions with final CXR: 16,930

✓ Ready to merge CXR reports with cohort


In [12]:
print(f"\n{'='*60}")
print("MERGING DISCHARGE NOTES AND CHEST X-RAYS")
print(f"{'='*60}")

# Start with cohort that already has discharge notes
print(f"Starting with cohort + discharge notes: {len(cohort_with_discharge):,}")

# Merge with CXR reports (LEFT JOIN - keep all admissions)
cohort_with_notes = cohort_with_discharge.merge(
    cxr_final[['hadm_id', 'radiology_text', 'radiology_charttime', 'radiology_note_id']],
    on='hadm_id',
    how='left'
)

print(f"\nFinal dataset: {len(cohort_with_notes):,} admissions")

# Check coverage
print(f"\n{'='*60}")
print("COVERAGE STATISTICS")
print(f"{'='*60}")
print(f"Admissions with discharge notes: {cohort_with_notes['discharge_text'].notna().sum():,} ({cohort_with_notes['discharge_text'].notna().sum()/len(cohort_with_notes)*100:.1f}%)")
print(f"Admissions with CXR reports: {cohort_with_notes['radiology_text'].notna().sum():,} ({cohort_with_notes['radiology_text'].notna().sum()/len(cohort_with_notes)*100:.1f}%)")
print(f"Admissions with BOTH notes: {(cohort_with_notes['discharge_text'].notna() & cohort_with_notes['radiology_text'].notna()).sum():,} ({(cohort_with_notes['discharge_text'].notna() & cohort_with_notes['radiology_text'].notna()).sum()/len(cohort_with_notes)*100:.1f}%)")
print(f"Admissions with discharge ONLY: {(cohort_with_notes['discharge_text'].notna() & cohort_with_notes['radiology_text'].isna()).sum():,}")

# Readmission rates
print(f"\n{'='*60}")
print("READMISSION RATES BY NOTE AVAILABILITY")
print(f"{'='*60}")

both_notes = (cohort_with_notes['discharge_text'].notna() & cohort_with_notes['radiology_text'].notna())
print(f"Admissions with BOTH notes:")
print(f"  Count: {both_notes.sum():,}")
print(f"  Readmission rate: {cohort_with_notes[both_notes]['readmitted_within_window'].mean()*100:.2f}%")

discharge_only = (cohort_with_notes['discharge_text'].notna() & cohort_with_notes['radiology_text'].isna())
print(f"\nAdmissions with discharge ONLY:")
print(f"  Count: {discharge_only.sum():,}")
print(f"  Readmission rate: {cohort_with_notes[discharge_only]['readmitted_within_window'].mean()*100:.2f}%")

print(f"\nOriginal cohort readmission rate: {cohort['readmitted_within_window'].mean()*100:.2f}%")


MERGING DISCHARGE NOTES AND CHEST X-RAYS
Starting with cohort + discharge notes: 17,940

Final dataset: 17,940 admissions

COVERAGE STATISTICS
Admissions with discharge notes: 17,940 (100.0%)
Admissions with CXR reports: 16,930 (94.4%)
Admissions with BOTH notes: 16,930 (94.4%)
Admissions with discharge ONLY: 1,010

READMISSION RATES BY NOTE AVAILABILITY
Admissions with BOTH notes:
  Count: 16,930
  Readmission rate: 24.78%

Admissions with discharge ONLY:
  Count: 1,010
  Readmission rate: 19.70%

Original cohort readmission rate: 24.49%


In [13]:
print(f"\n{'='*60}")
print("TEXT LENGTH ANALYSIS")
print(f"{'='*60}")

# Calculate text lengths (only for non-null texts)
cohort_with_notes['discharge_length'] = cohort_with_notes['discharge_text'].str.len()
cohort_with_notes['radiology_length'] = cohort_with_notes['radiology_text'].str.len()

# For rows with both texts
both_notes_mask = (cohort_with_notes['discharge_text'].notna() & cohort_with_notes['radiology_text'].notna())
cohort_with_notes['combined_length'] = 0
cohort_with_notes.loc[both_notes_mask, 'combined_length'] = (
    cohort_with_notes.loc[both_notes_mask, 'discharge_length'] +
    cohort_with_notes.loc[both_notes_mask, 'radiology_length']
)

print(f"\nDischarge note statistics (characters):")
print(cohort_with_notes['discharge_length'].describe())

print(f"\nRadiology report statistics (characters):")
print(cohort_with_notes[cohort_with_notes['radiology_text'].notna()]['radiology_length'].describe())

print(f"\nCombined text statistics (admissions with BOTH notes):")
print(cohort_with_notes[both_notes_mask]['combined_length'].describe())

# Token estimation (rough: 4 chars per token)
print(f"\n{'='*60}")
print("ESTIMATED TOKEN COUNTS")
print(f"{'='*60}")
cohort_with_notes['estimated_tokens'] = (cohort_with_notes['combined_length'] / 4).astype(int)

print(f"\nEstimated tokens for combined text:")
print(cohort_with_notes[both_notes_mask]['estimated_tokens'].describe())

# Check 8192 token limit
token_limit = 8192
exceeds_limit = (cohort_with_notes['estimated_tokens'] > token_limit).sum()
print(f"\nAdmissions exceeding 8192 token limit: {exceeds_limit:,} ({exceeds_limit/both_notes_mask.sum()*100:.2f}%)")

print(f"\nAverage lengths:")
print(f"  Discharge: {cohort_with_notes['discharge_length'].mean():.0f} chars (~{cohort_with_notes['discharge_length'].mean()/4:.0f} tokens)")
print(f"  Radiology: {cohort_with_notes[cohort_with_notes['radiology_text'].notna()]['radiology_length'].mean():.0f} chars (~{cohort_with_notes[cohort_with_notes['radiology_text'].notna()]['radiology_length'].mean()/4:.0f} tokens)")
print(f"  Combined: {cohort_with_notes[both_notes_mask]['combined_length'].mean():.0f} chars (~{cohort_with_notes[both_notes_mask]['combined_length'].mean()/4:.0f} tokens)")


TEXT LENGTH ANALYSIS

Discharge note statistics (characters):
count    17940.00000
mean     17172.42330
std       6357.13448
min       2227.00000
25%      12698.75000
50%      16332.50000
75%      20507.25000
max      60381.00000
Name: discharge_length, dtype: float64

Radiology report statistics (characters):
count    16930.000000
mean      1028.285588
std       1072.405333
min         71.000000
25%        454.000000
50%        601.000000
75%        923.750000
max       7785.000000
Name: radiology_length, dtype: float64

Combined text statistics (admissions with BOTH notes):
count    16930.000000
mean     18346.519905
std       6531.530299
min       3176.000000
25%      13745.250000
50%      17492.000000
75%      21846.500000
max      62866.000000
Name: combined_length, dtype: float64

ESTIMATED TOKEN COUNTS

Estimated tokens for combined text:
count    16930.000000
mean      4586.258181
std       1632.884290
min        794.000000
25%       3436.000000
50%       4372.500000
75%      

In [14]:
print(f"\n{'='*60}")
print("SAVING FINAL DATASET")
print(f"{'='*60}")

# Select columns to save
columns_to_save = [
    # IDs
    'subject_id', 'hadm_id',

    # Temporal
    'admittime', 'dischtime',

    # Demographics
    'age_at_admit', 'gender', 'race',

    # Admission details
    'admission_type', 'discharge_location',
    'length_of_stay_days',

    # Clinical conditions
    'is_cardiorenal_long', 'has_acute_kidney_injury', 'has_heart_failure',
    'has_hyponatremia', 'has_posthemorrhagic_anemia', 'has_sepsis',
    'has_any_cardiorenal_sepsis', 'has_aki_and_hf',

    # Clinical notes
    'discharge_text', 'discharge_charttime', 'discharge_note_id',
    'radiology_text', 'radiology_charttime', 'radiology_note_id',

    # Text metrics
    'discharge_length', 'radiology_length', 'combined_length', 'estimated_tokens',

    # Labels
    'readmitted_within_window', 'readmission_gap_in_days'
]

# Save
cohort_with_notes[columns_to_save].to_csv(OUTPUT_FILE, index=False)

print(f"\n✓ Dataset saved to: {OUTPUT_FILE}")
print(f"\nFile statistics:")
print(f"  Rows: {len(cohort_with_notes):,}")
print(f"  Columns: {len(columns_to_save)}")
print(f"  File size: {OUTPUT_FILE.stat().st_size / (1024*1024):.1f} MB")

print(f"\n{'='*60}")
print("FINAL DATASET SUMMARY")
print(f"{'='*60}")
print(f"Total admissions: {len(cohort_with_notes):,}")
print(f"Admissions with discharge notes: {cohort_with_notes['discharge_text'].notna().sum():,} (100.0%)")
print(f"Admissions with CXR reports: {cohort_with_notes['radiology_text'].notna().sum():,} (94.4%)")
print(f"Admissions with BOTH notes: {(cohort_with_notes['discharge_text'].notna() & cohort_with_notes['radiology_text'].notna()).sum():,} (94.4%)")

print(f"\nReadmission statistics:")
print(f"  Overall rate: {cohort_with_notes['readmitted_within_window'].mean()*100:.2f}%")
print(f"  Readmitted: {cohort_with_notes['readmitted_within_window'].sum():,}")
print(f"  Not readmitted: {(~cohort_with_notes['readmitted_within_window']).sum():,}")

print(f"\nText statistics:")
print(f"  Average discharge note: {cohort_with_notes['discharge_length'].mean():.0f} chars (~{cohort_with_notes['discharge_length'].mean()/4:.0f} tokens)")
print(f"  Average radiology report: {cohort_with_notes[cohort_with_notes['radiology_text'].notna()]['radiology_length'].mean():.0f} chars (~{cohort_with_notes[cohort_with_notes['radiology_text'].notna()]['radiology_length'].mean()/4:.0f} tokens)")
print(f"  Average combined: {cohort_with_notes[(cohort_with_notes['discharge_text'].notna()) & (cohort_with_notes['radiology_text'].notna())]['combined_length'].mean():.0f} chars (~{cohort_with_notes[(cohort_with_notes['discharge_text'].notna()) & (cohort_with_notes['radiology_text'].notna())]['combined_length'].mean()/4:.0f} tokens)")
print(f"  Notes exceeding 8K token limit: {(cohort_with_notes['estimated_tokens'] > 8192).sum():,} (3.1%)")

print(f"\n{'='*60}")
print("✓ NOTEBOOK 2 COMPLETE")
print(f"{'='*60}")
print(f"\nReady for Notebook 4B: Encode clinical notes with BioClinical-ModernBERT")


SAVING FINAL DATASET

✓ Dataset saved to: /content/drive/MyDrive/MIDS/w266/Final Project/output/long_los_cohort_with_notes.csv

File statistics:
  Rows: 17,940
  Columns: 30
  File size: 315.2 MB

FINAL DATASET SUMMARY
Total admissions: 17,940
Admissions with discharge notes: 17,940 (100.0%)
Admissions with CXR reports: 16,930 (94.4%)
Admissions with BOTH notes: 16,930 (94.4%)

Readmission statistics:
  Overall rate: 24.49%
  Readmitted: 4,394
  Not readmitted: 13,546

Text statistics:
  Average discharge note: 17172 chars (~4293 tokens)
  Average radiology report: 1028 chars (~257 tokens)
  Average combined: 18347 chars (~4587 tokens)
  Notes exceeding 8K token limit: 520 (3.1%)

✓ NOTEBOOK 2 COMPLETE

Ready for Notebook 4B: Encode clinical notes with BioClinical-ModernBERT


In [15]:
# Create summary table
print(f"\n{'='*60}")
print("COHORT PROGRESSION SUMMARY")
print(f"{'='*60}")

summary_data = {
    'Stage': [
        'Partner\'s original cohort',
        'After adding discharge notes',
        'After adding CXR reports',
        'Final cohort with BOTH notes'
    ],
    'Admissions': [
        len(cohort),
        cohort_with_notes['discharge_text'].notna().sum(),
        cohort_with_notes['radiology_text'].notna().sum(),
        (cohort_with_notes['discharge_text'].notna() & cohort_with_notes['radiology_text'].notna()).sum()
    ],
    'Readmission_Rate': [
        f"{cohort['readmitted_within_window'].mean()*100:.2f}%",
        f"{cohort_with_notes[cohort_with_notes['discharge_text'].notna()]['readmitted_within_window'].mean()*100:.2f}%",
        f"{cohort_with_notes[cohort_with_notes['radiology_text'].notna()]['readmitted_within_window'].mean()*100:.2f}%",
        f"{cohort_with_notes[(cohort_with_notes['discharge_text'].notna()) & (cohort_with_notes['radiology_text'].notna())]['readmitted_within_window'].mean()*100:.2f}%"
    ]
}

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

print(f"\nClinical condition prevalence (final cohort):")
condition_cols = [col for col in cohort_with_notes.columns if col.startswith('has_') or col.startswith('is_')]
for col in condition_cols:
    count = cohort_with_notes[col].sum()
    pct = count / len(cohort_with_notes) * 100
    print(f"  {col}: {count:,} ({pct:.1f}%)")


COHORT PROGRESSION SUMMARY
                       Stage  Admissions Readmission_Rate
   Partner's original cohort       17940           24.49%
After adding discharge notes       17940           24.49%
    After adding CXR reports       16930           24.78%
Final cohort with BOTH notes       16930           24.78%

Clinical condition prevalence (final cohort):
  is_cardiorenal_long: 10,204 (56.9%)
  has_acute_kidney_injury: 6,890 (38.4%)
  has_heart_failure: 4,928 (27.5%)
  has_hyponatremia: 2,963 (16.5%)
  has_posthemorrhagic_anemia: 3,073 (17.1%)
  has_sepsis: 3,183 (17.7%)
  has_any_cardiorenal_sepsis: 11,626 (64.8%)
  has_aki_and_hf: 2,922 (16.3%)
